In [1]:
import os
import pandas as pd
import datetime as dtime
import glob
import zipfile

In [ ]:
# Распаковка zip-файлов

os.chdir('C:/Users/sstarikov/Desktop/Python/USD_EUR Analysis')

# Working with zip
List_of_zip = [i for i in glob.glob('*.{}'.format('zip'))]

for zips in List_of_zip:
    unfold_zip = zipfile.ZipFile(zips)
    unfold_zip.extractall()
    os.rename('trades.csv', 'trades' + zips[25:37])
    unfold_zip.close()

In [64]:
# Working with csv
trades_appended = pd.DataFrame()
List_of_files = [i for i in glob.glob('*.{}'.format('csv'))]

for files in List_of_files:
    trades = pd.read_csv(files, sep=';', header = 1)
    trades_secid = trades['SECID']=='CNYRUB_TOM'
    trades_boardid = trades['BOARDID']=='CETS'
    trades_time1 = trades['TRADETIME']<='18:40:00'
    trades_time2 = trades['TRADETIME']>='18:30:00'
    trades_filter = trades[trades_boardid & trades_secid & trades_time1 & trades_time2]
    trades_appended = trades_appended.append(trades_filter)
    
trades_appended.to_csv('tradesALL_CNY.csv', sep = ';', index = False)

C:\ProgramData\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3165: DtypeWarning: Columns (8) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [65]:
trades = pd.read_csv('tradesALL_CNY.csv', sep = ';', decimal = ',')
trades['TRADEDATE'] = pd.to_datetime(trades['TRADEDATE'], format='%d.%m.%Y')
trades.drop(['BOARDID', 'BUYSELL', 'TRADENO'], axis = 1, inplace = True)


MIN = trades.groupby(['TRADEDATE'])['PRICE'].min().reset_index()
dfMIN = pd.DataFrame(MIN)
dfMIN = dfMIN.rename(columns = {'PRICE':'MIN'})

MAX = trades.groupby(['TRADEDATE'])['PRICE'].max().reset_index()
dfMAX = pd.DataFrame(MAX)
dfMAX = dfMAX.rename(columns = {'PRICE':'MAX'})

VOLCUR_SUM = trades.groupby(['TRADEDATE'])['VOLCUR'].sum().reset_index()
dfSUM_VOLCUR = pd.DataFrame(VOLCUR_SUM)
dfSUM_VOLCUR = dfSUM_VOLCUR.rename(columns = {'VOLCUR':'SUM(VOLCUR)'})

INVCURVOL_SUM = trades.groupby(['TRADEDATE'])['INVCURVOL'].sum().reset_index()
dfSUM_INVCURVOL = pd.DataFrame(INVCURVOL_SUM)
dfSUM_INVCURVOL = dfSUM_INVCURVOL.rename(columns = {'INVCURVOL':'SUM(INVCURVOL)'})

dfSUM = dfSUM_VOLCUR.merge(dfSUM_INVCURVOL, how = 'left', on = 'TRADEDATE')


trades1 = trades.merge(dfMIN, how = 'left', on = 'TRADEDATE')
trades2 = trades1.merge(dfMAX, how = 'left', on = 'TRADEDATE')
trades3 = trades2.merge(dfSUM, how = 'left', on = 'TRADEDATE')


trades3['WA'] = trades3['SUM(INVCURVOL)'] / trades3['SUM(VOLCUR)']

Final = trades3.groupby(['TRADEDATE']).first().reset_index()
Final.drop(['SUM(INVCURVOL)', 'SUM(VOLCUR)', 'TRADETIME', 'PRICE', 'VOLCUR', 'INVCURVOL'], axis = 1, inplace = True)
Final

,TRADEDATE,TRADETIME,SECID,PRICE,VOLCUR,INVCURVOL
0,2019-01-11,18:38:21,CNYRUB_TOM,9.9026,8000,79220.8
1,2019-01-14,18:33:33,CNYRUB_TOM,9.9015,745000,7376617.5
2,2019-01-14,18:33:39,CNYRUB_TOM,9.9015,2000,19803.0
3,2019-01-18,18:38:20,CNYRUB_TOM,9.7600,40000,390400.0
4,2019-01-18,18:38:20,CNYRUB_TOM,9.7659,10000,97659.0


In [70]:
Final.to_csv('C:/Users/sstarikov/Desktop/Python/USD_EUR Analysis/CNY_TOM_2019_2.csv', sep=';', index=False, decimal = ',')